# Auditoria del experimento — Lab 2 NLP

Entregable de sustentacion. Consulta el MLflow Tracking Server desplegado en la EC2
(directamente, o via los endpoints `/audit/*` de la API) para mostrar de forma resumida
la trazabilidad completa: protocolo, runs presentados, contribucion por integrante y
modelo final desplegado.

Ejecutar despues de que la infraestructura Docker (`docker/docker-compose.yml`) este
levantada y accesible desde internet.

In [ ]:
import os
import requests

# Reemplazar por la URL publica de la API/MLflow desplegadas en la EC2.
API_BASE_URL = os.environ.get("LAB2_API_BASE_URL", "http://<EC2_PUBLIC_IP>:8000")
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://<EC2_PUBLIC_IP>:5000")

import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

## 1. Estado del servicio (`/health`)

In [ ]:
health = requests.get(f"{API_BASE_URL}/health", timeout=10).json()
health

## 2. Run de protocolo (`/audit/protocol`)

In [ ]:
protocol_resp = requests.get(f"{API_BASE_URL}/audit/protocol", timeout=10)
print(protocol_resp.status_code)
protocol_resp.json()

## 3. Todos los runs presentados (`/audit/runs`)

In [ ]:
runs_resp = requests.get(f"{API_BASE_URL}/audit/runs", timeout=10)
print(runs_resp.status_code)

import pandas as pd

runs_data = runs_resp.json()["runs"]
runs_df = pd.DataFrame([
    {
        "run_id": r["run_id"],
        "lab_experiment_id": r["tags"].get("lab_experiment_id"),
        "lab_stage": r["tags"].get("lab_stage"),
        "lab_member_id": r["tags"].get("lab_member_id"),
        "macro_f1_mean": r["metrics"].get("macro_f1_mean"),
        "macro_f1_std": r["metrics"].get("macro_f1_std"),
    }
    for r in runs_data
])
runs_df.sort_values("macro_f1_mean", ascending=False)

## 4. Contribucion por integrante (`/audit/contributions`)

In [ ]:
contributions_resp = requests.get(f"{API_BASE_URL}/audit/contributions", timeout=10)
print(contributions_resp.status_code)
contributions_resp.json()

## 5. Modelo desplegado y trazabilidad (`/audit/model`)

In [ ]:
model_resp = requests.get(f"{API_BASE_URL}/audit/model", timeout=10)
print(model_resp.status_code)
model_resp.json()

## 6. Prueba de inferencia (`/api/v1/predict`)

Validar que el `model_run_id` devuelto coincide con el run final identificado en
`/audit/model`.

In [ ]:
predict_resp = requests.post(
    f"{API_BASE_URL}/api/v1/predict",
    json={"text": ["this movie was great", "worst day ever"]},
    timeout=10,
)
print(predict_resp.status_code)
predict_resp.json()

## 7. Casos invalidos (deben responder 4xx sin resultados parciales)

In [ ]:
invalid_cases = [
    {"text": None},
    {"text": ""},
    {"text": "   "},
    {"text": []},
    {"text": ["a"] * 33},
    {"text": "a" * 1001},
    {"text": [123]},
]

for case in invalid_cases:
    resp = requests.post(f"{API_BASE_URL}/api/v1/predict", json=case, timeout=10)
    print(case, "->", resp.status_code)